# WP2a: Sentinel-2 Pre-processing (Optical Track)

**Owner: colleague**

Cloud masking, reflectance normalisation, and NDSI computation for all S2 scenes 2019–2024.
Output of this notebook feeds directly into `03a_classification_s2.ipynb`.

> **Dependency for SAR track:** once training polygons are digitised in `03a`, upload them
> to GEE as a FeatureCollection asset — that asset is the training input for the SVM in `03b`.

In [ ]:
import ee
import geemap
import sys
sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project
from src.preprocessing_s2 import preprocess_s2

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

## 2a.1 Load S2 collection

In [ ]:
START, END = '2019-01-01', '2024-12-31'

s2_raw = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
)
print('Raw S2 images:', s2_raw.size().getInfo())

## 2a.2 Cloud masking + NDSI

Applies QA60 cloud/cirrus mask, scales to surface reflectance [0–1], and adds the
NDSI band: `(B3 – B11) / (B3 + B11)`.

In [ ]:
s2 = s2_raw.map(preprocess_s2)
print('Processed S2 images:', s2.size().getInfo())

## 2a.3 Inspect a sample image

In [ ]:
sample = s2.first().clip(aoi)
print('Sample date:', sample.date().format('YYYY-MM-dd').getInfo())

Map = geemap.Map()
Map.centerObject(aoi, zoom=9)
Map.addLayer(sample, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'S2 True Colour')
Map.addLayer(
    sample.select('NDSI'),
    {'min': -0.5, 'max': 1.0, 'palette': ['#1a6faf', 'white']},
    'NDSI'
)
Map

## Notes

- Cloud cover threshold is set to **80 %** — tighten to 20–30 % for classification to avoid partially cloudy scenes.
- `NDSI ≥ 0.4` is the standard ice threshold; inspect the distribution before committing.
- A stricter cloud filter (`< 20 %`) will substantially reduce the available scenes — check coverage before tightening.